# Sorani (Kurdish) Voice Training - Piper TTS

Bu not defteri, Sorani (ckb) sesini Piper TTS'e eğitir. Üç veri kaynağı:
- `mozilla` — Mozilla Data Collective'den indirilen Common Voice ckb (önerilen, kotasız, gerçek kadın ses)
- `synthetic` — kurdishtts ile üretilmiş sentetik Sorani (`synthetic.zip`)
- `commonvoice_hf` — HuggingFace Common Voice (artık ckb içermediği için genelde çalışmaz)

Eğitim sonunda `model.onnx` + `model.onnx.json` üretilir; bilgisayarda `deploy/stt/models/`'e koyup TTS'i devreye alırsın.

Adımlar: 1) kurulum 2) veri indir/tar.gz yükle 3) kadın konuşmacı seç + hazırla 4) EĞİT 5) ONNX + indir

In [ ]:
# 0) GPU kontrol
!nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo "GPU yok"

In [ ]:
!pip install -q piper-training datasets
print("kuruldu")

## 2) Veri kaynağını seç

In [ ]:
# 2) VERİ KAYNAĞI
# "mozilla" | "synthetic" | "commonvoice_hf"
DATA_SOURCE = "mozilla"
TARGET_HOURS = 1.0
print("Kaynak:", DATA_SOURCE)

### (A) Mozilla Data Collective — Common Voice ckb

**Mozilla yolunu kullanmak için önce tarayıcıda şunları yap:**
1. [mozilladatacollective.com](https://mozilladatacollective.com) → ücretsiz hesap aç.
2. Central Kurdish (ckb) Common Voice veri kümesini bul:  
   `https://mozilladatacollective.com/datasets/cmu5wr35b00donq07jqzj68wa`  
   → **Download** (tarayıcı `.tar.gz` indirir, birkaç yüz MB).
3. Aşağıdaki hücrede bu `.tar.gz` dosyasını yükle.

Veri içinde `validated.tsv` + `clips/*.mp3` bulunur; `validated.tsv` cinsiyet (`gender`), konuşmacı (`client_id`) ve metin (`sentence`) içerir.

In [ ]:
# 2.A) Mozilla ckb tar.gz yükle ve aç (yol A)
from google.colab import files
import glob, os, tarfile, shutil
os.makedirs("data", exist_ok=True)

print("Mozilla'dan indirdiğin .tar.gz dosyasını yükle:")
up = files.upload()
for fn in up:
    if fn.endswith((".tar.gz", ".tgz", ".tar")):
        os.makedirs("cv_extract", exist_ok=True)
        with tarfile.open(fn, "r:*") as t:
            t.extractall("cv_extract")
        print("açıldı:", fn)
        
# cv-.../ckb klasörünü bul
import glob
ckb_dirs = glob.glob("cv_extract/**/ckb", recursive=True) + glob.glob("cv_extract/**/ckb/", recursive=True)
print("ckb klasörü:", ckb_dirs)
if not ckb_dirs:
    # aksi halde herhangi bir validated.tsv ara
    tsvs = glob.glob("cv_extract/**/validated.tsv", recursive=True)
    print("bulunan validated.tsv:", tsvs[:3])
    if tsvs:
        CV = os.path.dirname(os.path.dirname(tsvs[0]))
    else:
        raise SystemExit("validated.tsv bulunamadı — dosyanın Common Voice ckb olup olmadığını kontrol et")
else:
    CV = ckb_dirs[0].rstrip("/")
print("CV kök:", CV)

### Kadın Sorani konuşmacıyı seç ve Piper verisini hazırla

In [ ]:
# 3) validated.tsv oku → en çok kayıt veren kadın Sorani konuşmacıyı seç → wav+txt hazırla
import os, glob, csv, random
import librosa, soundfile as sf
import numpy as np

CV = os.environ.get("CV_DIR", "cv_extract")  # yukarıda bulunan kök
tsv = os.path.join(CV, "validated.tsv")
os.makedirs("data", exist_ok=True)

# validated.tsv oku
rows = []
with open(tsv, "r", encoding="utf-8") as f:
    rd = csv.DictReader(f, delimiter="\t")
    for r in rd:
        rows.append(r)
print("toplam kayıt:", len(rows))

# kadınları filtrele
if "gender" in rows[0]:
    female = [r for r in rows if r.get("gender") == "female"]
    print("kadın kayıt:", len(female))
    if len(female) == 0:
        print("kadın yok → tümü kullanılıyor")
        female = rows
else:
    female = rows

# en çok kayıt veren tek konuşmacı (tutarlı tek ses)
from collections import Counter
cnt = Counter(r["client_id"] for r in female)
top = cnt.most_common(1)[0][0]
print("seçilen konuşmacı:", top, "| kayıt:", cnt[top])
sub = [r for r in female if r["client_id"] == top]

# hedef süre (~3 sn / kayıt)
target = int(TARGET_HOURS * 3600 / 3)
sub = sub[:target]

clips = os.path.join(CV, "clips")
done = 0
for i, r in enumerate(sub):
    p = os.path.join(clips, r["path"])
    if not os.path.exists(p):
        continue
    audio, sr = librosa.load(p, sr=22050, mono=True)
    wav = f"data/sorani_{i:05d}.wav"
    sf.write(wav, audio, 22050)
    open(wav.replace(".wav", ".txt"), "w", encoding="utf-8").write(r["sentence"])
    done += 1
print("hazır:", done, "kayıt")

### (B) Sentetik — yedek yol
`DATA_SOURCE == synthetic` ise bu hücreyi çalıştır (`synthetic.zip` yükle).

In [ ]:
# 2.B) Sentetik zip yükle (yedek yol)
from google.colab import files
import glob, os, zipfile
os.makedirs("data", exist_ok=True)
up = files.upload()
for fn in up:
    if fn.endswith(".zip"):
        with zipfile.ZipFile(fn, "r") as z:
            z.extractall("data")
        print("açıldı:", fn)
print("data/wav:", len(glob.glob("data/*.wav")))

## 4) Piper verisini hazırla + EĞİT

In [ ]:
# 4) Piper metadata hazırla
!piper.train_prepare --dataset-dir data --sample-rate 22050 --language ckb --config-out config.json
print("config ok")

In [ ]:
# 5) EĞİT (Colab GPU ~1-3 saat)
!piper.train --config-dir . --run-name sorani-female
print("eğitim bitti")

In [ ]:
# 6) ONNX + indir
!piper.export_onnx --run-dir runs --output-dir output --dataset-dir data
!ls -la output/
from google.colab import files
import glob
for f in sorted(glob.glob("output/*")):
    print("indir:", f)
    files.download(f)